## SoLEXS Multi-Day Light Curve

Author: Prakhar Singh  
Affiliation: Aryabhatta Research Institute of Observational Sciences (ARIES), Nainital  
Email: prakhar@aries.res.in  

---

### Purpose
This script provides an automated and efficient pipeline to combine and visualize multiple SoLEXS light curve (`.lc.gz`) datasets across several days into a single, continuous plot.  
The output file is automatically named based on the **date range** of the available observations.

---

### Overview
- All SoLEXS daily datasets are stored as ZIP files inside one base directory.  
- Automatically extracts all ZIP files if not already extracted.  
- Recursively searches for `.lc.gz` light curve files in each folder.  
- Reads the count rate and corresponding UTC time for each observation.  
- Combines all available light curves into **one unified plot**. 
- Automatically determines the **earliest and latest observation dates** and names the output file accordingly.  
- Saves the combined plot as:  
  **`SoLEXS_Lightcurve_<StartDate>-<EndDate>.png`**  




In [3]:
import os
import glob
import gzip
import zipfile
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone
from astropy.io import fits

# --- Base directory where all SoLEXS data ZIPs are stored ---
base_dir = "/Users/prakhar/HelioWork/solexs_pipleine/"

# --- Folder to save all plots ---
all_plots_dir = os.path.join(base_dir, "SoLEXS_Lightcurve_plots")
os.makedirs(all_plots_dir, exist_ok=True)


def unzip_file(zip_path):
    extract_dir = zip_path.replace(".zip", "")
    if not os.path.exists(extract_dir):
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_dir)
        print(f"✔ Extracted: {os.path.basename(zip_path)}")
    else:
        print(f"→ Already extracted: {os.path.basename(zip_path)}")
    return extract_dir


def read_lightcurve_data(lc_filename):
    """Read SoLEXS .lc.gz file and return time (UTC), counts, and observation date."""
    with gzip.open(lc_filename, 'rb') as f:
        with fits.open(f) as lc_hdul:
            plot_times_UNIX = lc_hdul[1].data['TIME']
            plot_lcurve = lc_hdul[1].data['COUNTS']
            plot_times_UTC = [datetime.fromtimestamp(t, tz=timezone.utc) for t in plot_times_UNIX]
            obs_date = datetime.fromisoformat(lc_hdul[1].header['DATE-OBS'].strip())
            return plot_times_UTC, plot_lcurve, obs_date


def process_solexs_data(base_dir):
    """Extract all .lc.gz files, combine them, and make a single multi-day plot (same color)."""
    zip_files = glob.glob(os.path.join(base_dir, "*.zip"))
    if not zip_files:
        print("⚠ No ZIP files found. Exiting.")
        return

    print(f"Processing {len(zip_files)} ZIP file(s)...")

    # Step 1: Extract all zip files
    extracted_dirs = [unzip_file(zf) for zf in zip_files]

    # Step 2: Collect all .lc.gz files
    lc_files = []
    for d in extracted_dirs:
        lc_files.extend(glob.glob(os.path.join(d, "**/*.lc.gz"), recursive=True))
    lc_files = sorted(lc_files)

    if not lc_files:
        print("⚠ No .lc.gz files found. Exiting.")
        return

    print(f"Found {len(lc_files)} .lc.gz files.")

    # Step 3: Read and combine all lightcurves into one plot
    fig, ax = plt.subplots(figsize=(14, 7))
    color = "tab:blue"

    all_dates = []

    for lc_file in lc_files:
        times, counts, obs_date = read_lightcurve_data(lc_file)
        all_dates.append(obs_date.date())
        ax.plot(times, counts, color=color, lw=1.2, alpha=0.8)

    # Step 4: Determine date range for output file naming
    start_date = min(all_dates)
    end_date = max(all_dates)
    date_range_str = f"{start_date.strftime('%d%m%Y')}-{end_date.strftime('%d%m%Y')}"

    # Step 5: Format the plot
    ax.set_yscale('log')
    ax.set_xlabel('Time [UTC]', fontsize=14)
    ax.set_ylabel('Counts', fontsize=14)
    ax.tick_params(axis="both", which="major", labelsize=15)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%H:%M', tz=timezone.utc))
    ax.set_title(f"SoLEXS Light Curve ({start_date.strftime('%d %b %Y')} - {end_date.strftime('%d %b %Y')})",
                 fontsize=14, weight="bold")

    plt.tight_layout()

    # Step 6: Save combined plot with date range
    outname = os.path.join(all_plots_dir, f"SoLEXS_Lightcurve_{date_range_str}.png")
    plt.savefig(outname, dpi=300)
    plt.close(fig)

    print(f"✅ Combined lightcurve saved as:\n   {outname}")


# --- Run the process ---
process_solexs_data(base_dir)


Processing 3 ZIP file(s)...
→ Already extracted: AL1_SLX_L1_20251020_v1.0.zip
→ Already extracted: AL1_SLX_L1_20251021_v1.0.zip
→ Already extracted: AL1_SLX_L1_20251022_v1.0.zip
Found 3 .lc.gz files.
✅ Combined lightcurve saved as:
   /Users/prakhar/HelioWork/solexs_pipleine/SoLEXS_Lightcurve_plots/SoLEXS_Lightcurve_20102025-22102025.png
